# UTS MODUL PRAKTIKUM 2 BIG DATA
## Pemrosesan Terstruktur Menggunakan PySpark DataFrame API & Spark SQL

Nama: Isnaini Nur Hasanah

NIM: 52240005

Prodi: Sains Data 2024

Mata Kuliah: Pemrograman Untuk Big Data

## 0. Setup PySpark di Google Colab

In [ ]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("NYC_Taxi_PySpark") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.5


## 1. Unduh Dataset NYC Taxi



In [ ]:
!pip install -q gdown

import os

FILE_ID  = "1hMSx3JrY6uiNWRj4OnZ5vLZRT80AA_Cm"
DRIVE_URL = f"https://drive.google.com/uc?id={FILE_ID}"
CSV_PATH = "/content/yellow_tripdata_2015-01.csv"

if not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) < 1_000_000_000:
    !gdown "{DRIVE_URL}" -O "{CSV_PATH}"
else:
    print("File sudah ada, lewati unduhan.")

!ls -lh /content/yellow_tripdata_2015-01.csv
!head -3 /content/yellow_tripdata_2015-01.csv

Downloading...
From (original): https://drive.google.com/uc?id=1hMSx3JrY6uiNWRj4OnZ5vLZRT80AA_Cm
From (redirected): https://drive.google.com/uc?id=1hMSx3JrY6uiNWRj4OnZ5vLZRT80AA_Cm&confirm=t&uuid=74da6857-3ff8-48d0-b9db-9e6de0c16437
To: /content/yellow_tripdata_2015-01.csv
100% 1.99G/1.99G [00:24<00:00, 80.0MB/s]
-rw-r--r-- 1 root root 1.9G Nov  7 17:49 /content/yellow_tripdata_2015-01.csv
VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
2,2015-01-15 19:05:39,2015-01-15 19:23:42,1,1.59,-73.993896484375,40.750110626220703,1,N,-73.974784851074219,40.750617980957031,1,12,1,0.5,3.25,0,0.3,17.05
1,2015-01-10 20:33:38,2015-01-10 20:53:28,1,3.30,-74.00164794921875,40.7242431640625,1,N,-73.994415283203125,40.759109497070313,1,14.5,0.5,0.5,2,0,0.3,17.8


## 2. Ingestion Data dengan Explicit Schema

In [ ]:
# Periksa header asli file terlebih dahulu (harus 19 kolom untuk data 2015)
header_cols = spark.read.option("header", "true").csv("file://" + CSV_PATH).columns
print(len(header_cols), "kolom:", header_cols)

# Skema eksplisit (urutan mengikuti file yellow_tripdata_2015-01.csv),
schema = StructType([
    StructField("vendor_id", StringType(), True),
    StructField("pickup_datetime", StringType(), True),
    StructField("dropoff_datetime", StringType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("rate_code_id", StringType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("payment_type", StringType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])
assert len(header_cols) == len(schema.fields), "Jumlah kolom file tidak sama dengan skema!"

file_path = "file://" + CSV_PATH

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(file_path)

print("--- SKEMA SPARK DATAFRAME ---")
df_raw.printSchema()
print(f"Total Record Data: {df_raw.count():,}")

19 kolom: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RateCodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']
--- SKEMA SPARK DATAFRAME ---
root
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- rate_code_id: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: doubl

## 3. Pembersihan Data & Feature Engineering

In [ ]:
df_clean = df_raw.dropna(subset=["pickup_datetime", "total_amount"]) \
    .filter((F.col("trip_distance") > 0) & (F.col("total_amount") > 0))

df_transformed = df_clean \
    .withColumn("pickup_ts", F.to_timestamp("pickup_datetime", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("dropoff_ts", F.to_timestamp("dropoff_datetime", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("pickup_hour", F.hour("pickup_ts")) \
    .withColumn("trip_duration_min", F.round((F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60, 2))

df_transformed = df_transformed.cache()
df_transformed.createOrReplaceTempView("v_nyc_taxi")

print("--- CONTOH DATA TERTATA (5 BARIS) ---")
df_transformed.select("vendor_id", "pickup_hour", "trip_distance", "trip_duration_min", "total_amount").show(5)

--- CONTOH DATA TERTATA (5 BARIS) ---
+---------+-----------+-------------+-----------------+------------+
|vendor_id|pickup_hour|trip_distance|trip_duration_min|total_amount|
+---------+-----------+-------------+-----------------+------------+
|        2|         19|         1.59|            18.05|       17.05|
|        1|         20|          3.3|            19.83|        17.8|
|        1|         20|          1.8|            10.05|        10.8|
|        1|         20|          0.5|             1.87|         4.8|
|        1|         20|          3.0|            19.32|        16.3|
+---------+-----------+-------------+-----------------+------------+
only showing top 5 rows


## 4. Agregasi DataFrame API (Contoh Demonstrasi)

In [ ]:
df_hourly_summary = df_transformed.groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    ) \
    .sort(F.desc("total_trips"))

print("--- RINGKASAN PERJALANAN PER JAM ---")
df_hourly_summary.show(5)

--- RINGKASAN PERJALANAN PER JAM ---
+-----------+-----------+------------+-------------+
|pickup_hour|total_trips|avg_distance|total_revenue|
+-----------+-----------+------------+-------------+
|         19|     800980|       25.87|1.536148889E7|
|         18|     795133|         6.1|1.142531899E7|
|         20|     730153|         9.3|1.062730219E7|
|         21|     707689|       41.91|1.063815104E7|
|         22|     683197|        34.8|1.049449016E7|
+-----------+-----------+------------+-------------+
only showing top 5 rows


## 5. TUGAS MANDIRI MAHASISWA (SPARK DATAFRAME & SPARK SQL)
### Kategori A: Analisis Tren Waktu & Metrik Perjalanan (DataFrame API)

**Tugas A1 - Analisis Jam Sibuk Penumpang (Peak Pick-up Hours)**

In [ ]:
# A1: group by pickup_hour -> count, avg distance, sum revenue -> Top 5 jam tersibuk
df_a1 = df_transformed.groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    ) \
    .orderBy(F.desc("total_trips"))

print("--- TOP 5 JAM SIBUK ---")
df_a1.show(5)

--- TOP 5 JAM SIBUK ---
+-----------+-----------+------------+-------------+
|pickup_hour|total_trips|avg_distance|total_revenue|
+-----------+-----------+------------+-------------+
|         19|     800980|       25.87|1.536148889E7|
|         18|     795133|         6.1|1.142531899E7|
|         20|     730153|         9.3|1.062730219E7|
|         21|     707689|       41.91|1.063815104E7|
|         22|     683197|        34.8|1.049449016E7|
+-----------+-----------+------------+-------------+
only showing top 5 rows


Jam 19:00 adalah jam tersibuk dengan 800.980 perjalanan dan pendapatan sekitar 15,36 juta USD. Lima jam teratas semuanya jatuh pada 18:00–22:00, jadi permintaan taksi memuncak saat jam pulang kerja sampai malam. Pendapatan per jam mengikuti jumlah perjalanan, tetapi avg_distance tidak konsisten: 6,1 mil pada jam 18 melawan 25,87 mil (jam 19), 41,91 mil (jam 21), dan 34,8 mil (jam 22). Rata-rata sepanjang itu tidak masuk akal untuk taksi kota, dan penyebabnya kemungkinan besar outlier jarak yang salah catat (terbukti di C1). Jadi jumlah perjalanan dan pendapatan lebih bisa dipercaya daripada rata-rata jarak.

**Tugas A2 - Analisis Durasi Perjalanan Berdasarkan Jumlah Penumpang**

In [ ]:
# A2: group by passenger_count, filter > 0, urut berdasarkan jumlah penumpang
df_a2 = df_transformed.filter(F.col("passenger_count") > 0) \
    .groupBy("passenger_count") \
    .agg(
        F.round(F.avg("trip_duration_min"), 2).alias("avg_duration_min"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.count("*").alias("total_trips")
    ) \
    .orderBy("passenger_count")

print("--- DURASI & TARIF PER JUMLAH PENUMPANG ---")
df_a2.show()

--- DURASI & TARIF PER JUMLAH PENUMPANG ---
+---------------+----------------+----------------+-----------+
|passenger_count|avg_duration_min|avg_total_amount|total_trips|
+---------------+----------------+----------------+-----------+
|              1|           14.32|           15.07|    8927494|
|              2|           13.89|           15.31|    1805916|
|              3|           14.08|           14.89|     526099|
|              4|            13.8|           14.86|     252105|
|              5|           14.55|           14.87|     695449|
|              6|           14.08|           14.63|     453124|
|              7|           11.32|           17.05|          7|
|              8|           17.41|           39.17|          3|
|              9|           28.87|           84.88|          6|
+---------------+----------------+----------------+-----------+



Sekitar 70% perjalanan hanya berpenumpang 1 orang (8,93 juta dari sekitar 12,66 juta). Untuk 1–6 penumpang, durasi rata-rata nyaris sama (13,8–14,6 menit) dan tarif rata-rata juga sama (sekitar 14,6–15,3 USD), jadi jumlah penumpang praktis tidak memengaruhi durasi maupun tarif karena argo dihitung per jarak dan waktu, bukan per orang. Penumpang 7–9 hanya punya 7, 3, dan 6 perjalanan, sehingga tarif tinggi seperti 84,88 USD pada 9 penumpang tidak bisa dijadikan kesimpulan. Kelompok ini sebaiknya diabaikan atau dianggap anomali.

### Kategori B: Analisis SQL & Segmentasi Tarif (Spark SQL)

**Tugas B1 - Segmentasi Jarak Tempuh dengan CASE WHEN**

> Catatan: batas 'Medium' pada soal (`BETWEEN 2.1 AND 10.0`) menyisakan celah untuk jarak 2.0 < d < 2.1 (hasilnya NULL). Agar semua data tercakup, dipakai `> 2.0 AND <= 10.0`.

In [ ]:
df_b1 = spark.sql("""
    SELECT
        CASE
            WHEN trip_distance <= 2.0 THEN 'Short Trip'
            WHEN trip_distance > 2.0 AND trip_distance <= 10.0 THEN 'Medium Trip'
            ELSE 'Long Trip'
        END AS trip_segment,
        COUNT(*)                    AS total_trips,
        ROUND(AVG(fare_amount), 2)  AS avg_fare,
        ROUND(AVG(tip_amount), 2)   AS avg_tip
    FROM v_nyc_taxi
    GROUP BY 1
    ORDER BY avg_fare
""")

print("--- SEGMENTASI JARAK TEMPUH ---")
df_b1.show()

--- SEGMENTASI JARAK TEMPUH ---
+------------+-----------+--------+-------+
|trip_segment|total_trips|avg_fare|avg_tip|
+------------+-----------+--------+-------+
|  Short Trip|    7491192|     7.0|    0.9|
| Medium Trip|    4587365|   15.59|   2.88|
|   Long Trip|     587861|   44.88|   5.92|
+------------+-----------+--------+-------+



Perjalanan jarak pendek mendominasi: sekitar 59% (7,49 juta), diikuti sedang sekitar 36% (4,59 juta) dan jauh hanya sekitar 4,6% (0,59 juta). Tarif naik tajam seiring jarak, dari 7,00 ke 15,59 ke 44,88 USD, begitu pula tip dari 0,90 ke 2,88 ke 5,92 USD. Kalau dilihat sebagai rasio tip terhadap tarif, segmen sedang paling besar (sekitar 18%), sedangkan pendek (sekitar 13%) dan jauh (sekitar 13%) lebih rendah. Rata-rata tip ini juga masih bercampur dengan pembayaran tunai yang tipnya tidak tercatat (lihat B2).

**Tugas B2 - Analisis Metode Pembayaran & Tip Rate**

In [ ]:
df_b2 = spark.sql("""
    SELECT
        payment_type,
        COUNT(*) AS total_trips,
        ROUND(AVG((tip_amount / total_amount) * 100), 2) AS avg_tip_pct
    FROM v_nyc_taxi
    GROUP BY payment_type
    ORDER BY avg_tip_pct DESC
""")

print("--- RATA-RATA TIP (%) PER TIPE PEMBAYARAN ---")
df_b2.show()

print("--- TIPE PEMBAYARAN DENGAN RATA-RATA TIP TERTINGGI ---")
df_b2.limit(1).show()

--- RATA-RATA TIP (%) PER TIPE PEMBAYARAN ---
+------------+-----------+-----------+
|payment_type|total_trips|avg_tip_pct|
+------------+-----------+-----------+
|           1|    7853509|      15.15|
|           4|       8513|       0.02|
|           3|      29117|       0.01|
|           2|    4775278|        0.0|
|           5|          1|        0.0|
+------------+-----------+-----------+

--- TIPE PEMBAYARAN DENGAN RATA-RATA TIP TERTINGGI ---
+------------+-----------+-----------+
|payment_type|total_trips|avg_tip_pct|
+------------+-----------+-----------+
|           1|    7853509|      15.15|
+------------+-----------+-----------+



Tipe pembayaran 1 (kartu kredit) memberi rata-rata tip tertinggi, yaitu 15,15%, sedangkan tipe 2 (tunai) tercatat 0,00% meski jumlah perjalanannya besar (4,78 juta). Ini lebih mencerminkan cara pencatatan data daripada perilaku penumpang: sistem hanya merekam tip yang dibayar lewat kartu, tip tunai tidak masuk ke data. Tipe 3, 4, dan 5 (tanpa biaya, sengketa, tidak diketahui) jumlahnya kecil dan tipnya hampir nol, jadi tidak berpengaruh pada kesimpulan.

### Kategori C: Advanced Window Functions & Export

**Tugas C1 - Ranking Perjalanan Terjauh Per Jam (Window Function)**

In [ ]:
window_spec = Window.partitionBy("pickup_hour").orderBy(F.desc("trip_distance"))

df_c1 = df_transformed.withColumn("distance_rank", F.dense_rank().over(window_spec))

print("--- PERJALANAN TERJAUH (RANK 1) PADA JAM 08:00, 12:00, 18:00 ---")
df_c1.filter((F.col("distance_rank") == 1) & (F.col("pickup_hour").isin(8, 12, 18))) \
    .select("pickup_hour", "distance_rank", "pickup_datetime", "dropoff_datetime",
            "trip_distance", "trip_duration_min", "total_amount") \
    .orderBy("pickup_hour") \
    .show(truncate=False)

--- PERJALANAN TERJAUH (RANK 1) PADA JAM 08:00, 12:00, 18:00 ---
+-----------+-------------+-------------------+-------------------+-------------+-----------------+------------+
|pickup_hour|distance_rank|pickup_datetime    |dropoff_datetime   |trip_distance|trip_duration_min|total_amount|
+-----------+-------------+-------------------+-------------------+-------------+-----------------+------------+
|8          |1            |2015-01-25 08:15:49|2015-01-25 08:55:47|196.8        |39.97            |8.63        |
|12         |1            |2015-01-04 12:09:56|2015-01-04 12:29:17|66746.8      |19.35            |19.8        |
|18         |1            |2015-01-03 18:26:48|2015-01-03 18:35:48|2331800.0    |9.0              |8.8         |
+-----------+-------------+-------------------+-------------------+-------------+-----------------+------------+



Ketiga baris rank 1 jelas data tidak valid. Jam 08:00 mencatat 196,8 mil dalam 39,97 menit dengan total hanya 8,63 USD. Jam 12:00 mencatat 66.746,8 mil dalam 19,35 menit, dan jam 18:00 mencatat 2.331.800 mil dalam 9 menit. Kecepatan dan tarifnya mustahil, jadi ini kesalahan pencatatan argo atau GPS, bukan perjalanan terjauh yang sebenarnya. Temuan ini juga menjelaskan avg_distance yang janggal di A1. Pada analisis sungguhan, data perlu difilter dengan batas wajar, misalnya jarak maksimum tertentu atau kecepatan rata-rata yang masuk akal, sebelum agregasi.

**Tugas C2 - Export Hasil Agregasi (A1) ke Parquet**

In [ ]:
output_path = "/content/output/nyc_hourly_summary.parquet"

# Simpan seluruh hasil agregasi A1 (24 jam), bukan hanya Top 5
df_a1.write.mode("overwrite").parquet(output_path)

# Verifikasi
!ls -lh /content/output/nyc_hourly_summary.parquet
spark.read.parquet(output_path).orderBy(F.desc("total_trips")).show(5)

total 4.0K
-rw-r--r-- 1 root root 1.9K Nov  7 18:03 part-00000-699a5559-9e01-4789-8238-d5382badae3d-c000.snappy.parquet
-rw-r--r-- 1 root root    0 Nov  7 18:03 _SUCCESS
+-----------+-----------+------------+-------------+
|pickup_hour|total_trips|avg_distance|total_revenue|
+-----------+-----------+------------+-------------+
|         19|     800980|       25.87|1.536148889E7|
|         18|     795133|         6.1|1.142531899E7|
|         20|     730153|         9.3|1.062730219E7|
|         21|     707689|       41.91|1.063815104E7|
|         22|     683197|        34.8|1.049449016E7|
+-----------+-----------+------------+-------------+
only showing top 5 rows


Hasil agregasi tersimpan dengan benar: ada file part-00000-...snappy.parquet dan penanda _SUCCESS, yang menandakan penulisan selesai tanpa error. Ukurannya hanya 1,9 KB karena isinya cuma 24 baris (satu per jam), dan format Parquet yang terkompresi snappy membuatnya sangat kecil. Data yang dibaca ulang identik dengan hasil A1 (jam 19, 18, 20, 21, 22 dengan angka yang sama), jadi proses tulis-baca berjalan konsisten.

In [ ]:
spark.stop()